# Lesson 4: Token and position embeddings

Lesson 3 prepared batches as integer token IDs with shape `(B, T)`. A Transformer needs a vector at each position. GPT-2 looks up a learned token vector and a learned position vector, adds them, then sends the result into its first Transformer block.

We will make those lookups visible with tiny tables: batch size `B=2`, sequence length `T=3`, and embedding width `C=2`. The Python, C, and Java examples share the same IDs and table values. GPT-2 uses the same operations with far larger learned tables: token embeddings `50,257 × 768` and position embeddings `1,024 × 768`.

This is the lookup and addition underlying `torch.nn.Embedding`; no large tensor library is needed to see how the indices and dimensions line up. The real implementation is in Karpathy's [`GPT.forward`](../../build-nanogpt/train_gpt2.py).

**Read the companions:** [C embeddings_lesson.c](../../c/src/embeddings_lesson.c) and [Java EmbeddingsLesson.java](../../java/src/main/java/learning/karpathy/gpt2/EmbeddingsLesson.java). Run with `-Action lesson -Lesson 4`.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "gpt-2", *Path.cwd().parents]
tutorial_root = next(
    (candidate for candidate in candidates
     if (candidate / "experiments/configs/tiny-playground.json").is_file()),
    None,
)
if tutorial_root is None:
    raise FileNotFoundError("Open the repository, gpt-2 folder, or its notebook folder and restart the kernel.")

from gpt2_lab.config import ModelConfig

gpt2 = ModelConfig.gpt2_124m()
print(f"Python: {sys.executable}")
print(f"GPT-2 parameter tables: WTE=({gpt2.vocab_size:,}, {gpt2.n_embd}), WPE=({gpt2.context_length:,}, {gpt2.n_embd})")
print("The example below uses much smaller tables so every lookup fits on screen.")


## 1. The input is a grid of token IDs

Each row is one sequence in the batch. The column tells us its position in that sequence. Here `B=2` and `T=3`; the position IDs `[0, 1, 2]` are shared across both rows.

An embedding table is a matrix of learned vectors. A token ID selects a row of the token table. A position ID selects a row of the position table.


In [ ]:
BATCH_SIZE = 2
SEQUENCE_LENGTH = 3
EMBEDDING_SIZE = 2

token_ids = [[1, 2, 3], [3, 2, 1]]
position_ids = list(range(SEQUENCE_LENGTH))

print(f"Token IDs (B, T) = ({len(token_ids)}, {len(token_ids[0])}): {token_ids}")
print(f"Position IDs (T) = ({len(position_ids)},): {position_ids}")


## 2. Look up token vectors

The small token table below stands in for GPT-2's learned matrix `WTE`. Indexing with a token ID retrieves a vector of width `C`. Applying this to every ID changes the shape from `(B, T)` to `(B, T, C)`.

For example, token ID `2` retrieves row 2, `[0.0, 1.0]`. The values here are chosen for visibility; training learns the real values.


In [ ]:
token_embedding_table = [
    [0.0, 0.0],  # ID 0
    [1.0, 0.0],  # ID 1
    [0.0, 1.0],  # ID 2
    [1.0, 1.0],  # ID 3
]

token_vectors = [
    [token_embedding_table[token_id] for token_id in row]
    for row in token_ids
]
print(f"Token vectors (B, T, C) = ({BATCH_SIZE}, {SEQUENCE_LENGTH}, {EMBEDDING_SIZE}):")
for batch_index, sequence in enumerate(token_vectors):
    print(f"  batch {batch_index}: {sequence}")


## 3. Add a position vector at every sequence location

Token identity alone does not say whether a token appeared first or third. GPT-2 uses a learned position table `WPE`; each sequence location selects one of its rows. That same position row is added to every batch item at that location.

The sum has the same shape `(B, T, C)` as the token vectors. It is the initial residual stream, passed into Transformer block 0. Later lessons will show how attention and the MLP update that stream.


In [ ]:
position_embedding_table = [
    [0.1, 0.0],  # position 0
    [0.0, 0.2],  # position 1
    [0.3, 0.3],  # position 2
]

input_vectors = [
    [
        [token_vectors[batch][position][channel]
         + position_embedding_table[position][channel]
         for channel in range(EMBEDDING_SIZE)]
        for position in range(SEQUENCE_LENGTH)
    ]
    for batch in range(BATCH_SIZE)
]

print("Position vectors (T, C):", position_embedding_table)
print("Initial residual stream (B, T, C):")
for batch_index, sequence in enumerate(input_vectors):
    print(f"  batch {batch_index}: {sequence}")

assert input_vectors[0] == [[1.1, 0.0], [0.0, 1.2], [1.3, 1.3]]
assert input_vectors[1] == [[1.1, 1.0], [0.0, 1.2], [1.3, 0.3]]


## Connect this to `GPT.forward`

The reference creates `pos = [0, ..., T-1]`, looks up `wpe(pos)` and `wte(idx)`, and adds the results. The position lookup has shape `(T, C)` while the token lookup has shape `(B, T, C)`. PyTorch broadcasts the position vectors across the batch dimension; the C and Java loops make that reuse explicit.

At the production GPT-2 size, `idx` has shape `(B,T)`, `wte(idx)` and the sum have shape `(B,T,768)`, and `wpe(pos)` has shape `(T,768)`. The model accepts sequences up to `T=1,024`.

**Try it:** reverse a row of `token_ids`. The same token IDs receive different combined vectors at different positions. Then change a row in the position table and see that it affects both batch items at that column.

**Reference:** [`GPT.forward` and the `wte`/`wpe` definitions](../../build-nanogpt/train_gpt2.py#L79).
